## EDA Statsbomb

Ziel dieses Notebooks ist es, zwei Kennzahlen für das Statistik-Tab des GSA zu entwickeln, die auf die Quelle **Statsbomb** zugreifen und zur aktuellen Architektur des Tabs passen. 

Hinweis: Dieses Notebook wird mit Gemini und Claude entwickelt.

----------------------------------

Bisherige Architektur (siehe Notebook `EDA_Openliga_DB_minimal.ipynb`)
```

┌─────────────────────────────────────┐
│  Filter: [Saison ▾]  [Verein ▾]      │
└─────────────────────────────────────┘

┌──────────┬──────────┬──────────┐
│ Platz    │ Punkte   │ Siege    │
│   3      │   68     │   21     │
└──────────┴──────────┴──────────┘

┌──────────┬──────────┐
│ Tore     │ Gegentore│
│   89     │   32     │
└──────────┴──────────┘

[Barchart: Tore vs. Gegentore über alle Vereine]



--------------------------------

```
┌─────────────────────────────────────┐
│  Filter: [Saison ▾]  [Verein ▾]      │
└─────────────────────────────────────┘

┌──────────┬──────────┬──────────┐
│ Platz    │ Punkte   │ Siege    │
│   3      │   68     │   21     │
└──────────┴──────────┴──────────┘

┌─────────────────────┬─────────────────────┐
│ Tore (OpenLigaDB)   │ Gegentore           │
│   89 (xG: 82.4)     │   32                │
├─────────────────────┼─────────────────────┤
│ Chancen-Verwertung  │ Druck-Resistenz     │
│   +6.6 (Top-Wert)   │   76.4 % (StatsBomb)│
└─────────────────────┴─────────────────────┘

[Barchart: Tore vs. Gegentore über alle Vereine]

### Erklärung der neuen Kennzahlen (xG-Effizienz und Druck-Resistenz)

Die Angaben "Tore" und "Gegentore" (OpenLigaDB) werden durch "xG-Effizienz" und "Druck-Resistenz" ergänzt (StatsBomb-Daten). Diese bieten eine taktische Ursachenanalyse über eine gesamte Bundesliga-Saison (34 Spieltage) hinweg.

---

#### 1. Chancen-Qualität & Tor-Effizienz

Der mathematische Wert **Expected Goals (xG)** misst die historische Wahrscheinlichkeit, mit der ein Schuss aus einer bestimmten Position (Winkel, Distanz, Gegnerdruck) im Tor landet. Ein Wert von `0.1` bedeutet beispielsweise, dass dieser Schuss statistisch gesehen in 10 % der Fälle zu einem Tor führt.

Für die Saison-Ebene nutzen wir die **Tor-Effizienz** ($\Delta\text{xG}$):
$$\text{Tor-Effizienz} = \text{Tatsächlich erzielte Tore} - \text{Erwartbare Tore (Gesamt-xG)}$$

###### Interpretation für die Saison-Analyse:
Da über 34 Spieltage hinweg kurzfristiges Glück mathematisch weitgehend herausgefiltert wird, zeigt diese Kennzahl die echte Qualität der Offensive:

* **Positiver Wert (z. B. $+6.6$):** *Extrem effizient.* Das Team kann schwierige Chancen verwerten oder behält in Schlüsselmomenten sehr gut die Nerven.
* **Wert um Null (z. B. $\pm0.0$):** *Präzise im Soll.* Das Team erzielt exakt so viele Tore, wie es sich laut Qualität der Chancen statistisch erarbeitet hat.
* **Negativer Wert (z. B. $-5.1$):** *Chancenwucher / Abschluss-Schwäche.* Das Team erarbeitet sich zwar hochkarätige Chancen, vergibt diese aber unterdurchschnittlich oft.

---

#### 2. Druck-Resistenz (Pressure Resistance)

StatsBomb erfasst jede Aktion, in der ein Spieler vom Gegner unter direkten physischen oder taktischen Druck gesetzt wird (*Pressure Event*). Die bloße Anzahl dieser Situationen ist über eine Saison hinweg zu hoch, um aussagekräftig zu sein.

Daher die Kennzahl **Druck-Resistenz (%)**:
$$\text{Druck-Resistenz} = \left( \frac{\text{Erfolgreiche Folge-Aktionen unter Druck}}{\text{Gegnerische Drucksituationen gesamt}} \right) \times 100$$

###### Interpretation für die Saison-Analyse:
Diese Metrik liefert die taktische Erklärung, die direkt unter der **Gegentore-Kachel** platziert wird:

- **Hohe Druck-Resistenz (z. B. $>75\%$):** Das Team kann sich auch in engen Räumen spielerisch befreien. Es verliert im Spielaufbau selten den Ball in der eigenen Hälfte, was die Anzahl der gefährlichen gegnerischen Konter und somit die Gegentore drastisch minimiert.
- **Niedrige Druck-Resistenz:** Das Team gerät bei gegnerischem Pressing in Panik, produziert Fehlpässe im defensiven Drittel und lädt den Gegner zu einfachen Torchancen ein.

---

Zusammenhang zu den beiden anderen Tabs:
1. **Verbindung zu Tab 2 (Clustering):** Teams mit ähnlicher Druck-Resistenz landen oft im selben spielerischen Cluster (z. B. "Ballbesitz-dominierte Teams" vs. "Konter-Teams").
2. Gewonnene Kennzahl kann auch für die Integration in das Chattab interessant sein.

------------------------------------

## Umsetzung der beiden neuen Kennzahlen fürs Statistik-Tab

Vorgehensweise: Aus den verschachtelten json-Daten von Statsbomb zwei "Typen" raussuchen:
- **Schüsse (Shot)**: Jedes Mal, wenn geschossen wird, gibt StatsBomb die erzielten goals (0 oder 1) und den eingebauten statsbomb_xg-Wert (eine Zahl zwischen 0 und 1).
- **Drucksituationen (Pressure)**: Jedes Mal, wenn ein Spieler bedrängt wird, herausfinden, ob das direkt darauffolgende Event desselben Teams erfolgreich war (kein Fehlpass, kein Ballverlust).

### Setup

In [10]:
import pandas as pd
from statsbombpy import sb

### 1. Schritt: Oberste Ebene der verschachtelten Datenstruktur: Wettbewerbe

- Leitfrage: Welche Daten zur 1. Bundesliga sind verfügbar? Welche Saisonen kann man kostenlos abrufen?

In [5]:
# Alle Wettbewerbe abrufen
all_comps = sb.competitions()

# Nur nach der deutschen 1. Bundesliga filtern
bundesliga_seasons = all_comps[all_comps['competition_name'] == '1. Bundesliga']

print("Verfügbare Bundesliga-Saisons bei StatsBomb:")
display(bundesliga_seasons[['competition_id', 'season_id', 'season_name']])

Verfügbare Bundesliga-Saisons bei StatsBomb:


c:\Users\Annette\00_Data_Science\06_NLP_GenAI\05_sandbox_task_2\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,season_name
0,9,281,2023/2024
1,9,27,2015/2016


Frei zugänglich gibt es nur Daten für die Saison 2015/16 (nicht relevant) und für 2023/24 (relevant, da im Filter des Statistik-Tab enthalten). Wählt der Nutzer über die Filter eine andere Saison, muss ein Strich für die Angaben "Chancen-Verwertung" und "Druck-Resistenz" erscheinen. 

----------------------------

### 2. Schritt: Spiele zur Saison 23/24 rausfiltern

Um dem User im Dashboard eine Auswahlliste (Dropdown) mit den richtigen Vereinen anzubieten, müssen zuerst alle Spiele der ausgewählten Saison geholt werden und daraus die Teamnamen herausgefiltert werden.

In [6]:
# 1. Wir holen uns alle Spiele für die Bundesliga (9) in der Saison 23/24 (281)
matches_23_24 = sb.matches(competition_id=9, season_id=281)

# 2. Sammeln aller Vereine, die entweder ein Heimspiel oder ein Auswärtsspiel hatten
heim_teams = matches_23_24['home_team'].unique()
auswaerts_teams = matches_23_24['away_team'].unique()

# 3. Beide Listen zusammenführen, Duplikate löschen (mit set) und alphabetisch sortieren
alle_vereine = sorted(list(set(heim_teams) | set(auswaerts_teams)))

print(f"Gefundene Vereine für die Saison 23/24 ({len(alle_vereine)} Teams):")
print(alle_vereine)

c:\Users\Annette\00_Data_Science\06_NLP_GenAI\05_sandbox_task_2\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Gefundene Vereine für die Saison 23/24 (18 Teams):
['Augsburg', 'Bayer Leverkusen', 'Bayern Munich', 'Bochum', 'Borussia Dortmund', 'Borussia Mönchengladbach', 'Darmstadt 98', 'Eintracht Frankfurt', 'FC Heidenheim', 'FC Köln', 'FSV Mainz 05', 'Freiburg', 'Hoffenheim', 'RB Leipzig', 'Union Berlin', 'VfB Stuttgart', 'Werder Bremen', 'Wolfsburg']


Schreibweise der Vereine beachten bzw. anpassen, damit der Filter später funktioniert! 

- Beispiel: Bayern Munich soll Bayern München heißen!
- Abgleich mit den Daten aus OpenligaDB wichtig!
- Lösung: Übersetzer-Dictionary

In [7]:
# Das Vereins-Mapping: "OpenLigaDB-Name": "StatsBomb-Name"
TEAM_NAME_MAPPING = {
    "FC Bayern München": "Bayern Munich",
    "Bayer 04 Leverkusen": "Bayer Leverkusen",
    "Borussia Dortmund": "Borussia Dortmund",
    "VfB Stuttgart": "VfB Stuttgart",
    "RB Leipzig": "RB Leipzig",
    "Eintracht Frankfurt": "Eintracht Frankfurt",
    "TSG 1899 Hoffenheim": "TSG Hoffenheim",
    "1. FC Heidenheim": "Heidenheim",
    "SV Werder Bremen": "Werder Bremen",
    "SC Freiburg": "Freiburg",
    "FC Augsburg": "Augsburg",
    "VfL Wolfsburg": "Wolfsburg",
    "1. FSV Mainz 05": "Mainz",
    "Borussia Mönchengladbach": "Borussia Mönchengladbach",
    "1. FC Union Berlin": "Union Berlin",
    "VfL Bochum": "Bochum",
    "1. FC Köln": "FC Cologne",
    "SV Darmstadt 98": "Darmstadt"
}

---------------------------------

### KPI Chancen-Verwertung bauen
- inhaltlicher Zusammenhang zu Toren beachten!
- Datenbezug aus soccer.db
- Ziel: SQL-Query üben
- Berechnung:

$$\text{Tor-Effizienz} = \text{Tatsächlich erzielte Tore} - \text{Erwartbare Tore (Gesamt-xG)}$$

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/soccer.db")


Mit Pandas:

In [4]:
query = """
SELECT *
FROM statsbomb_events
WHERE type = 'Shot'
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)
df

,id,match_id,index,period,minute,second,type,possession,possession_team,play_pattern,...,shot_technique,shot_body_part,pass_length,pass_angle,pass_outcome,pass_recipient,loc_x,loc_y,league,season_label
0,ddd816bd-6e0b-4b35-a838-b4c11d319a25,3890563,279,1,7,26,Shot,20,Ingolstadt,From Free Kick,...,Half Volley,Right Foot,None,None,None,None,98.0,58.6,1. Bundesliga,2015/16
1,3a9b7d9e-6294-4bd0-bc41-acb99b05272c,3890563,311,1,8,13,Shot,22,Ingolstadt,From Counter,...,Normal,Right Foot,None,None,None,None,100.1,42.3,1. Bundesliga,2015/16
2,e7790d14-c223-45d2-b3c3-437c1e5c7ac8,3890563,461,1,11,51,Shot,31,Ingolstadt,Regular Play,...,Normal,Head,None,None,None,None,112.2,32.7,1. Bundesliga,2015/16
3,1565e311-3882-46b7-9141-4b904309f6bc,3890563,614,1,15,17,Shot,40,Ingolstadt,Regular Play,...,Normal,Left Foot,None,None,None,None,114.2,34.2,1. Bundesliga,2015/16
4,7a049e7b-2158-4af5-9ac3-a81300a5b848,3890563,887,1,22,51,Shot,55,Ingolstadt,From Free Kick,...,Volley,Right Foot,None,None,None,None,113.4,36.2,1. Bundesliga,2015/16
5,ca2eb15f-d14e-4479-a89c-fc2bb63bb628,3890563,903,1,24,3,Shot,58,Ingolstadt,From Free Kick,...,Normal,Head,None,None,None,None,106.8,39.8,1. Bundesliga,2015/16
6,ea9079d4-541f-42f6-a656-b8fa17785c47,3890563,937,1,24,53,Shot,60,Ingolstadt,Regular Play,...,Half Volley,Right Foot,None,None,None,None,95.5,56.5,1. Bundesliga,2015/16
7,2027e467-9be2-4660-bfe2-b6fb786b787e,3890563,1126,1,30,56,Shot,73,Bayer Leverkusen,From Throw In,...,Normal,Right Foot,None,None,None,None,110.9,32.3,1. Bundesliga,2015/16
8,6841ff94-a6c9-43cd-a858-cf322acd99e2,3890563,1295,1,36,11,Shot,81,Bayer Leverkusen,From Free Kick,...,Normal,Right Foot,None,None,None,None,106.7,48.3,1. Bundesliga,2015/16
9,77cf0509-98e5-463a-9f4e-a43db079a0e2,3890563,1379,1,40,6,Shot,86,Ingolstadt,From Free Kick,...,Half Volley,Right Foot,None,None,None,None,109.6,61.4,1. Bundesliga,2015/16


Ohne Pandas:

In [5]:
query = """
SELECT *
FROM statsbomb_events
WHERE type = 'Shot'
LIMIT 10;
"""

rows = conn.execute(query).fetchall()
rows[:10]

[('ddd816bd-6e0b-4b35-a838-b4c11d319a25',
  3890563,
  279,
  1,
  7,
  26,
  'Shot',
  20,
  'Ingolstadt',
  'From Free Kick',
  'Ingolstadt',
  'Moritz Hartmann',
  'Right Wing',
  1.238041,
  None,
  None,
  0.008072011,
  'Off T',
  'Half Volley',
  'Right Foot',
  None,
  None,
  None,
  None,
  98.0,
  58.6,
  '1. Bundesliga',
  '2015/16'),
 ('3a9b7d9e-6294-4bd0-bc41-acb99b05272c',
  3890563,
  311,
  1,
  8,
  13,
  'Shot',
  22,
  'Ingolstadt',
  'From Counter',
  'Ingolstadt',
  'Pascal Groß',
  'Right Center Midfield',
  0.045288,
  None,
  None,
  0.04667507,
  'Blocked',
  'Normal',
  'Right Foot',
  None,
  None,
  None,
  None,
  100.1,
  42.3,
  '1. Bundesliga',
  '2015/16'),
 ('e7790d14-c223-45d2-b3c3-437c1e5c7ac8',
  3890563,
  461,
  1,
  11,
  51,
  'Shot',
  31,
  'Ingolstadt',
  'Regular Play',
  'Ingolstadt',
  'Mathew Leckie',
  'Left Wing',
  1.36054,
  None,
  None,
  0.09706584,
  'Off T',
  'Normal',
  'Head',
  None,
  None,
  None,
  None,
  112.2,
  32.7,


-------------------------------------------

### 1. Schritt: xG abrufen

In [7]:
query = """
SELECT
    team,
    shot_statsbomb_xg,
    shot_outcome
FROM statsbomb_events
WHERE type = 'Shot'
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)
df

,team,shot_statsbomb_xg,shot_outcome
0,Ingolstadt,0.008072,Off T
1,Ingolstadt,0.046675,Blocked
2,Ingolstadt,0.097066,Off T
3,Ingolstadt,0.977242,Goal
4,Ingolstadt,0.267620,Off T
5,Ingolstadt,0.029614,Off T
6,Ingolstadt,0.018916,Wayward
7,Bayer Leverkusen,0.365020,Goal
8,Bayer Leverkusen,0.439847,Goal
9,Ingolstadt,0.006357,Saved


-------------------------------------

### 2. Schritt: xG-Summe pro Team berechnen
Das ist der erste wichtige Aggregationsschritt:
- SUM(...) addiert alle xG-Werte
- GROUP BY team macht daraus eine Zeile pro Team

In [10]:
query = """
SELECT
    team,
    SUM(shot_statsbomb_xg) AS xg_summe
FROM statsbomb_events
WHERE type = 'Shot'
GROUP BY team
ORDER BY xg_summe DESC;
"""

df = pd.read_sql_query(query, conn)
df

,team,xg_summe
0,Bayer Leverkusen,122.493973
1,Barcelona,74.093792
2,Leicester City,66.282171
3,Arsenal,65.087769
4,Napoli,64.630668
...,...,...
78,Union Berlin,0.583984
79,Celta Vigo,0.494430
80,Hannover 96,0.330485
81,Osasuna,0.167656


-------------------------------------

### 3. Schritt: Tatsächliche Tore

Jetzt kommen die tatsächlich erzielten Tore dazu. In StatsBomb sind Tore keine eigene Zahl in einer extra Tabelle, sondern Schüsse mit `shot_outcome = 'Goal'`.     
Idee:
- Jeder Schuss, dessen `shot_outcome` gleich Goal ist, zählt als 1 Tor.
- Alle anderen zählen als 0.

In [9]:
query = """
SELECT
    team,
    SUM(shot_statsbomb_xg) AS xg_summe,
    SUM(CASE WHEN shot_outcome = 'Goal' THEN 1 ELSE 0 END) AS tore
FROM statsbomb_events
WHERE type = 'Shot'
GROUP BY team
ORDER BY xg_summe DESC;
"""

df = pd.read_sql_query(query, conn)
df

,team,xg_summe,tore
0,Bayer Leverkusen,122.493973,137
1,Barcelona,74.093792,102
2,Leicester City,66.282171,67
3,Arsenal,65.087769,62
4,Napoli,64.630668,78
...,...,...,...
78,Union Berlin,0.583984,0
79,Celta Vigo,0.494430,0
80,Hannover 96,0.330485,0
81,Osasuna,0.167656,1


------------------------------------

### 4. Schritt: Differenz bauen

Interpretation:

- positive differenz: mehr Tore erzielt als nach xG erwartet
- negative differenz: weniger Tore erzielt als nach xG erwartet

In [11]:
query = """
SELECT
    team,
    SUM(shot_statsbomb_xg) AS xg_summe,
    SUM(CASE WHEN shot_outcome = 'Goal' THEN 1 ELSE 0 END) AS tore,
    SUM(CASE WHEN shot_outcome = 'Goal' THEN 1 ELSE 0 END) - SUM(shot_statsbomb_xg) AS differenz
FROM statsbomb_events
WHERE type = 'Shot'
GROUP BY team
ORDER BY differenz DESC;
"""

df = pd.read_sql_query(query, conn)
df

,team,xg_summe,tore,differenz
0,Barcelona,74.093792,102,27.906208
1,AS Roma,61.949056,83,21.050944
2,Bayer Leverkusen,122.493973,137,14.506027
3,Napoli,64.630668,78,13.369332
4,Juventus,61.679564,72,10.320436
...,...,...,...,...
78,West Bromwich Albion,38.626782,32,-6.626782
79,Carpi,41.047142,33,-8.047142
80,Watford,44.082146,36,-8.082146
81,Hellas Verona,42.945356,34,-8.945356


-----------------------------------------